# Setup

In [5]:
%matplotlib inline
import pandas as pd
import geopandas as gpd

In [6]:
import rasterio
import rasterio.plot
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.geometry import Polygon
from descartes import PolygonPatch
import shapely.geometry
import numpy as np
import fiona
from rasterio.mask import raster_geometry_mask
from ws3.common import hash_dt
try:
   import cPickle as pickle
except:
   import pickle

Compiled interpolation routine not available
Compiled sparse grid routine not available


Read in a list of TSA basenames (will need this later).

In [7]:
def read_basenames(path):
    return [line.lower().strip().split(' ')[0] 
        for line in open(path, 'r') if not line.startswith('#')]
basenames = read_basenames('../dat/basenames.txt')

# Recent BC wildfire spatial netdown

Our DEG inventory has been updated to 2015. Wildfires were exceptionally intense in 2015-2017 (10x historical burn rate), so we decided to patch the 2015 inventory to net out these recent fires. To implement this, we will overlay spatial vector data from recent fires over the rasterized inventory data, and flip THEME1 for the affected pixels to 0 (i.e., remove the area from the THLB).


## Load spatial vector data describing recent fires

We have two datasets: one for 2015 fires, and one for 2016-2017 fires. We import only geometry and fire severity data.

First, we concatenate the two datasets into a single GeoDataFrame. Then, we need to clean up the fire severity data values (different encoding in each dataset). Finally, we need to filter the merged data to keep only the high-severity fires. 

In [8]:
gdf2 = gpd.read_file('../dat/gis/bc_fires_2015.gdb')[['geometry', 'SeverityClass']]
gdf3 = gpd.read_file('../dat/gis/bc_fires_2016-2017.gdb')[['geometry', 'SeverityClass']]

In [9]:
gdf = pd.concat([gdf2, gdf3])

Print fire severity class values for each dataset.

In [10]:
gdf2.SeverityClass.unique()

array(['Unburned', 'Low', 'Medium', 'High', 'Unknown'], dtype=object)

In [11]:
gdf3.SeverityClass.unique()

array(['Not', 'High', 'Medium', 'Low', 'Nuked'], dtype=object)

Clean up fire-severity values.

In [12]:
gdf.loc[gdf.SeverityClass == 'Not', 'SeverityClass'] = 'Unburned'
gdf.loc[gdf.SeverityClass == 'Unknown', 'SeverityClass'] = 'Unburned'
gdf.loc[gdf.SeverityClass == 'Nuked', 'SeverityClass'] = 'High'
gdf.SeverityClass.unique()

array(['Unburned', 'Low', 'Medium', 'High'], dtype=object)

Now that fire severity attributes are cleaned up, filter out all but high severity files.

In [13]:
gdf = gdf.query("SeverityClass in ['High']")

Compile list of GeoJSON-like features from fire GeoDataFrame geometry.

In [14]:
features = [shapely.geometry.mapping(f) for f in gdf.geometry]

## Patch raster inventory data

The following chunk of code loops through each TSA basename, opens the rasterized GeoTIFF inventory file for that TSA, and uses `rasterio.mask.mask` function to spatially overlay vector fire and raster inventory data. Affected inventoy pixels are processed sequentially. Note that only pixels with THEME1 value of '1' (i.e., in the THLB) are updated.

Updated copies of both hdt hash map pickle files and raster inventory GeoTIFF files are saved back to disk. 

**Note that all of these operations are performed on copies of the production data files (in `../dat/tif_firepatch`). The production dataset needs to be manually patched after running this notebook.**

In [16]:
N = []
for bn in basenames:
    with rasterio.open('../dat/firepatch/%s.tif' % bn, 'r+') as src:
        hdt = pickle.load(open('../dat/firepatch/hdt_%s.pkl' % bn, 'rb'))
        x = src.read()
        out_image, out_transform = rasterio.mask.mask(src, features)
        ix = np.where(out_image[1] > 0)
        H = []
        n = 0
        for i, _ in enumerate(ix[0]):
            ix_ = (ix[0][i:i+1], ix[1][i:i+1])
            h = x[0][ix_][0]
            dtk = list(hdt[h])
            if dtk[1] == '1':
                dtk[1] = '0'
                dtk_ = tuple(dtk)
                h_ = hash_dt(dtk_)
                if h_ in hdt:
                    assert hdt[h_] == dtk_
                else:
                    hdt[h_] = dtk_
                n += 1
                x[0][ix_] = h_ # update band-1 pixels to NTHLB dtk
                x[1][ix_] = 0  # update band-2 pixels to 0 (reset age after fire)
        N.append(n)
        print(bn, 'updated', n, 'of', ix[0].shape[0], 'candidate pixels')
        pickle.dump(hdt, open('../dat/firepatch/hdt_%s.pkl' % bn, 'wb'))
        src.write(x[0], indexes=1)
        src.write(x[1], indexes=2)

tsa01 updated 0 of 342 candidate pixels
tsa02 updated 0 of 142 candidate pixels
tsa03 updated 0 of 0 candidate pixels


/usr/local/lib/python3.6/dist-packages/rasterio/mask.py:88: UserWarning: shapes are outside bounds of raster. Are they in different coordinate reference systems?
  warnings.warn('shapes are outside bounds of raster. '


tsa04 updated 0 of 0 candidate pixels
tsa05 updated 0 of 8281 candidate pixels
tsa07 updated 0 of 2054 candidate pixels
tsa08 updated 0 of 11764 candidate pixels
tsa09 updated 0 of 4439 candidate pixels
tsa10 updated 0 of 0 candidate pixels
tsa11 updated 0 of 4242 candidate pixels
tsa12 updated 0 of 0 candidate pixels
tsa13 updated 0 of 2109 candidate pixels
tsa14 updated 0 of 65 candidate pixels
tsa15 updated 0 of 83 candidate pixels
tsa16 updated 0 of 547 candidate pixels
tsa17 updated 0 of 327 candidate pixels
tsa18 updated 0 of 890 candidate pixels
tsa19 updated 0 of 1651 candidate pixels
tsa20 updated 0 of 0 candidate pixels
tsa21 updated 0 of 5 candidate pixels
tsa22 updated 0 of 3293 candidate pixels
tsa23 updated 0 of 11018 candidate pixels
tsa24 updated 0 of 1500 candidate pixels
tsa25 updated 0 of 0 candidate pixels
tsa26 updated 0 of 19755 candidate pixels
tsa27 updated 0 of 1 candidate pixels
tsa29 updated 0 of 77996 candidate pixels
tsa30 updated 0 of 821 candidate pixels


## Patch harvest volume targets

In the previous step, we switched some inventory pixels from THLB to NTHLB state. Now, we need to reduce TSA-wise harvest volume target values to reflect the smaller harvesting landbase. To do this, we need to calculate the proportion of THLB in each TSA that was affected by the fire patch in the previous step.

In [17]:
df_patched = pd.DataFrame({'tsa':basenames, 'n_patched':N}).set_index('tsa')
df_targets = pd.read_csv('../dat/targets.csv').set_index('tsa')
df_patched['a_thlb'] = pd.read_csv('../dat/firepatch/tsa_area.csv').set_index('tsa')['a_thlb']
df_patched['p_netdown'] = df_patched.n_patched / df_patched.a_thlb

In [18]:
for bn in basenames:
    df_targets.at[bn, 'vcut_patched'] = df_targets.at[bn, 'vcut'] * (1. - df_patched.at[bn, 'p_netdown'])

Estimate relative impact of fire netdown patch on province-wide harvest volume.

In [19]:
df_targets.vcut_patched = df_targets.vcut_patched.astype(int)

In [20]:
(df_targets.vcut - df_targets.vcut_patched).sum() / df_targets.vcut.sum()

0.0

In [21]:
df_targets.vcut = df_targets.vcut_patched
df_targets = df_targets.drop('vcut_patched', axis=1)

Save a copy of the updated `targets.csv` file to disk.

In [22]:
df_targets.to_csv('../dat/firepatch/targets.csv')